In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
data_path = os.getenv("Data_path_clean")

In [3]:
df=pd.read_csv(data_path)
df.head()

,CustomerID,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer,42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
x=df.drop(['Churn','CustomerID'],axis=1)
y=df["Churn"]

In [5]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42,stratify=y)

In [6]:
print(x_train["TotalCharges"].max())
print(x_train["TotalCharges"].min())

8684.8
0.0


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler

In [8]:
x.columns

Index(['Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='object')

In [9]:
binary_colm = ['Gender', 'SeniorCitizen', 'Partner', 'Dependents','PhoneService', 'MultipleLines', 
               'OnlineSecurity', 'OnlineBackup','DeviceProtection', 
               'TechSupport', 'StreamingTV', 'StreamingMovies','PaperlessBilling']
multi_cat_colm = ['InternetService', 'PaymentMethod', 'Contract']
numerical_colm = ['Tenure', 'MonthlyCharges', 'TotalCharges']

y_train = y_train.map({'Yes': 1, 'No': 0})
y_test = y_test.map({'Yes': 1, 'No': 0})

In [10]:
binary_pipe = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])

multi_cat_pipe = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))])

numerical_pipe = Pipeline(steps=[
    ('scaler', RobustScaler())])

In [11]:
preprocessor = ColumnTransformer(transformers=[
    ('bin', binary_pipe, binary_colm),
    ('multi', multi_cat_pipe, multi_cat_colm),
    ('num', numerical_pipe, numerical_colm)])

In [12]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

In [18]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum())}

In [20]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    pipe.fit(x_train, y_train)
    
    y_pred = pipe.predict(x_test)
    y_proba = pipe.predict_proba(x_test)[:, 1]
    
    print(f"\n--- {name} ---")
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


--- Logistic Regression ---
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

ROC-AUC: 0.8414
Confusion Matrix:
 [[745 290]
 [ 81 293]]

--- Random Forest ---
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC: 0.8233
Confusion Matrix:
 [[925 110]
 [187 187]]

--- XGBoost ---
              precision    recall  f1-score   support

           0       0.86      0.78      0.82      1035
           1       0.52      0.66      0.58       37

In [22]:
import joblib

best_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])
best_pipe.fit(x_train, y_train)

model_path = os.path.join('E:\coding\PROJECTS\customer_churn_prediction\models', 'churn_model_logreg.pkl')
joblib.dump(best_pipe, model_path)

<>:9: SyntaxWarning: invalid escape sequence '\c'
<>:9: SyntaxWarning: invalid escape sequence '\c'
C:\Users\hunnu\AppData\Local\Temp\ipykernel_37144\347731455.py:9: SyntaxWarning: invalid escape sequence '\c'
  model_path = os.path.join('E:\coding\PROJECTS\customer_churn_prediction\models', 'churn_model_logreg.pkl')


['E:\\coding\\PROJECTS\\customer_churn_prediction\\models\\churn_model_logreg.pkl']